# sgRNA library quality control

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Loading and filtering data

Input: whitespace-delimited count matrix containing one row per sgRNA. The first columns identify the sgRNA and its target, while the remaining columns contain counts for the experimental samples and the two libraries.

Only rows assigned to PA locus tags or `IntergBefore_PA` regions are retained. Other guide categories are excluded from the subsequent quality-control summaries.

In [2]:
# Colors
blue = "#466dab"
orange = "#e38024"
red = "#e32424"
lightgrey = 'lightgrey'

In [ ]:
sgrna_counts_file = "path/to/sgrna_counts.txt"

# Read the sgrna file into pandas dataframe
df = pd.read_csv(sgrna_counts_file, sep='\s+')
print(f"Number of rows in the original file: {df.shape[0]} rows.")

# Keep only rows with sgRNA for PA genes or intergenic regions
dfpa = df[(df['Gene'].str.startswith('IntergBefore_PA')) | (df['Gene'].str.startswith('PA'))]
print(f"Number of rows with only PA or IntergBefore_PA: {dfpa.shape[0]} rows.")

In [ ]:
# Show columns names
dfpa.columns[2:]

In [ ]:
dfpa.head()

## 2. Distribution of sequencing reads in the initial libraries.

In [7]:
# Create a df for each library and sort it on counts descending
df_amp = dfpa[['Gene', 'LB.init_2_all___pBx_Amp_lib_']].copy().sort_values(by='LB.init_2_all___pBx_Amp_lib_', ascending=False)
df_gm = dfpa[['Gene', 'LB.init_1_all___pBx_Gm_lib_']].copy().sort_values(by='LB.init_1_all___pBx_Gm_lib_', ascending=False)


In [8]:
# Change here the style of the plot 
def make_sg_plot(*, y, name):
    x = range(1,y.shape[0]+1)
    plt.figure(figsize=(4,4))
    plt.plot(x, y, color=blue)
    # Change here the alpha for the filling
    plt.fill_between(x, y, color=blue, alpha=1)
    plt.xlim(left=0)
    plt.yscale('log') 
    plt.ylabel('sgRNA counts')
    plt.title(name)
    plt.tight_layout()
    plt.savefig(f"{name}_sgplot.svg")
    plt.show()

In [ ]:
make_sg_plot(y=df_amp['LB.init_2_all___pBx_Amp_lib_'], name='Amp library')
make_sg_plot(y=df_gm['LB.init_1_all___pBx_Gm_lib_'], name='Gm library')

![Ampicillin library read distribution](figures/Amp%20library_sgplot.svg)

![Gentamicin library read distribution](figures/Gm%20library_sgplot.svg)

## 3. Count distributions across samples

Horizontal boxplots compare sgRNA count distributions across the 48-hour experimental replicates and the two initial libraries. 

In [ ]:
# Choose the columns for the boxplot
for_box_plot = dfpa[['LBni_48h_Rep1', 'LBni_48h_Rep2', 'LBni_48h_Rep3',
       'LBni_48h_Rep4', 'LB_48h_Rep1', 'LB_48h_Rep2', 'LB_48h_Rep3',
       'LB_48h_Rep4', 'Glc_48h_Rep1', 'Glc_48h_Rep2', 'Glc_48h_Rep3',
       'Glc_48h_Rep4', 'Succ_48h_Rep1', 'Succ_48h_Rep2', 'Succ_48h_Rep3',
       'Succ_48h_Rep4', 'LB.init_1_all___pBx_Gm_lib_',
       'LB.init_2_all___pBx_Amp_lib_']].copy()

x_names = ['LBni_48h_Rep1', 'LBni_48h_Rep2', 'LBni_48h_Rep3',
       'LBni_48h_Rep4', 'LB_48h_Rep1', 'LB_48h_Rep2', 'LB_48h_Rep3',
       'LB_48h_Rep4', 'Glc_48h_Rep1', 'Glc_48h_Rep2', 'Glc_48h_Rep3',
       'Glc_48h_Rep4', 'Succ_48h_Rep1', 'Succ_48h_Rep2', 'Succ_48h_Rep3',
       'Succ_48h_Rep4', 'Gm_lib_',
       'Amp_lib_']

# Calculate the 10th and 90th percentiles
percentiles_10 = [for_box_plot[col_name].quantile(0.10) for col_name in for_box_plot.columns]
percentiles_90 = [for_box_plot[col_name].quantile(0.90) for col_name in for_box_plot.columns]

# Create the boxplot
plt.figure(figsize=(8,5))
sns.boxplot(data=for_box_plot, color=lightgrey, log_scale=True, orient='h')
for i in range(len(for_box_plot.columns)):
    p10 = percentiles_10[i]
    p90 = percentiles_90[i]
    # Here change the style for the percentiles
    plt.plot([p10, p10], [i-0.3, i+0.3], color='black', linestyle='-')
    plt.plot([p90, p90], [i-0.3, i+0.3], color='black', linestyle='-')

plt.xlabel('Number of counts')
plt.yticks(list(range(len(x_names))), x_names)
plt.tight_layout()
plt.savefig('Boxplots_all_cond.svg')

![Count distributions across samples](figures/Boxplots_all_cond.svg)

## 4. Per-sample summary statistics

For each count column, the functions below report the number and percentage of zero counts, non-zero counts, sgRNAs below 10 counts, and sgRNAs with at least 10 counts. They also calculate the minimum, maximum, median, and selected quantiles.

Two CSV summaries are generated: one for all count columns and one restricted to the 48-hour samples and initial libraries used in the boxplot.

In [11]:
def describe_ser(ser):
    """
    Compute the statistics for a Series and return them as a Series
    """
    number_of_na =  ser.isna().sum()
    total_counts = ser.count()
    number_of_0 = (ser == 0).sum()
    number_of_non_0 = (ser != 0).sum()
    number_of_sgrna_with_less_than_10 = (ser < 10).sum()
    number_of_sgrna_with_10_or_more = (ser >= 10).sum()
 
    
    data = {
        "total_counts": total_counts,
        "number_of_0": number_of_0,
        "number_of_non_0":  number_of_non_0,
        "number_of_sgrna_with_less_than_10": number_of_sgrna_with_less_than_10,
        "number_of_sgrna_with_10_or_more": number_of_sgrna_with_10_or_more,
        "number_of_0_%": round((number_of_0/total_counts)*100, 2),
        "number_of_non_0_%": round((number_of_non_0/total_counts)*100, 2),
        "number_of_sgrna_with_less_than_10_%": round((number_of_sgrna_with_less_than_10/total_counts)*100, 2),
        "number_of_sgrna_with_10_or_more_%": round((number_of_sgrna_with_10_or_more/total_counts)*100, 2),
        "min": ser.min(),
        "q2.5": ser.quantile(0.025),
        "q5": ser.quantile(0.05),
        "q10": ser.quantile(0.1),
        "q25": ser.quantile(0.25),
        "q50": ser.quantile(0.5),
        "q75": ser.quantile(0.75),
        "q90": ser.quantile(0.9),
        "q95": ser.quantile(0.95),
        "q97.5": ser.quantile(0.975),
        "max": ser.max()
    }
    results = pd.Series(data)
    
    results.name = ser.name
    #print(results)     
    return results

def describe_df(df, name):
    data_dict = {col_name: describe_ser(df[col_name]) for col_name in df.columns}
    df_with_stats = pd.DataFrame(data_dict)
    df_with_stats.to_csv(f"stats_sgRNA_{name}.csv")
    return df_with_stats

In [ ]:
describe_df(dfpa.iloc[:,2:], "all")

In [ ]:
describe_df(for_box_plot, '48h')

## 5. Gene representation and gene-length distributions

This section asks whether genes that are poorly represented in an initial library differ in length from well-represented genes. Intergenic targets are excluded before the guide table is joined to the genome annotation.

For this analysis, a gene is considered **valid** when it has at least three associated sgRNAs with at least 10 counts in the selected initial library. Genes with fewer than three qualifying sgRNAs are classified as **not valid**. The overlaid histograms compare gene-length distributions between these two groups.

In [22]:
# Count how many genes have at least 3 sgRNA with 10 counts

gm = 'LB.init_1_all___pBx_Gm_lib_'
amp = 'LB.init_2_all___pBx_Amp_lib_'

# Load the genome file 
GENOME_FILE = "../PAO1_107.csv"
genome = pd.read_csv(GENOME_FILE)

def make_validity_df(*, df, library, genome):

    # Keep only genes, not intergenic regions
    df_genes = df[(df['Gene'].str.startswith('PA'))].copy()

    # Add a columns with number of sgRNA per gene
    df_genes['number_of_sgrna'] = df_genes.groupby('Gene')['Gene'].transform("count")

    # Add a column that indicates for each sgRNA True if the corresponding gene has at least 3 sgRNA and the nb of counts is at least 10
    df_genes['valid_sgRNA'] = ((df_genes['number_of_sgrna'] >= 3) & (df_genes[library] >= 10))

    # Add a column with nb of valid sgRNA for each gene (if it has )at least 3 True in the valid_sgRNA col )
    df_genes['valid_gene'] = df_genes.groupby('Gene')['valid_sgRNA'].transform("sum")

    # Keep only one row per gene in the original df 
    df_genes = df_genes.drop_duplicates(subset='Gene')

    # Merge the df with the genome file to get gene length
    genes = df_genes.merge(genome, how='left', left_on='Gene', right_on='Locus Tag')

    nb_valid = (genes["valid_gene"] >= 3).sum()

    nb_not_valid = (genes["valid_gene"] < 3).sum()

    print(f"Number of valid genes: {int(nb_valid)}\nNumber of not valid genes: {int(nb_not_valid)} ")
    
    # Define valid and not valid groups

    genes_not_valid = genes[genes["valid_gene"] < 3]
    genes_valid = genes[genes["valid_gene"] >= 3]


    plt.figure(figsize=(5, 5))

    sns.histplot(genes_valid['Length (nucleotides)'], color=orange, label='valid', kde=True, log_scale=True)
    sns.histplot(genes_not_valid['Length (nucleotides)'], color=blue, label='not valid', kde=True, log_scale=True)

    plt.savefig(f"library_{library}_hist.svg")

    return genes

In [ ]:
df_valid_amp = make_validity_df(df=dfpa, library=amp, genome=genome)

![Gene-length distribution for the ampicillin library](figures/Amp_hist.svg)

In [ ]:
df_valid_gm = make_validity_df(df=dfpa, library=gm, genome=genome)

![Gene-length distribution for the gentamicin library](figures/Gm_hist.svg)

In [ ]:
df_valid_amp.sort_values(by='Gene')

## 6. Genes absent from the sgRNA library

Finally, the set of gene-targeting sgRNAs is compared with the genome annotation. Locus tags present in the genome annotation but absent from the sgRNA library are written to `missing_genes.csv`. 

In [ ]:
# Make missing gene list (csv file)
dff = dfpa[(dfpa['Gene'].str.startswith('PA'))].copy()
dff= dff.drop_duplicates(subset='Gene')
missing = genome.merge(dff, how='outer', left_on='Locus Tag', right_on='Gene')
missings = missing[['Locus Tag', 'Gene']]
missing_genes = missings[missings['Gene'].isna()]
missing_genes.to_csv('missing_genes.csv', index=False)
print(f"Number of missing genes: {genome.shape[0] - dff.shape[0]}.")